# GT-Free OCR Metrics - Dataset Exploration

This notebook demonstrates the two companion datasets released with the paper
**"GT-Free OCR Metrics: Reference-Free Evaluation via Render-and-Compare"**
(NeurIPS 2026 Evaluations & Datasets Track).

| Dataset | HuggingFace repo (parquet, recommended) | Raw per-page layout |
|---|---|---|
| Render-and-Compare pairs | `gt-free-ocr-metrics/omnidocbench-render-compare-parquet` | `gt-free-ocr-metrics/omnidocbench-render-compare` |
| Qwen OCR Log-Probabilities | `gt-free-ocr-metrics/omnidocbench-qwen-ocr-logprobs-parquet` | `gt-free-ocr-metrics/omnidocbench-qwen-ocr-logprobs` |

The **parquet editions** are recommended - image PNG bytes and JSON content
are stored inline, so a few HTTP requests download the whole dataset and there
are no HuggingFace rate-limit issues. The raw per-page repos are retained for
backward compatibility.

**Runtime:** all cells complete in under 2 minutes (downloads ~20 MB of sample data).
**Colab:** click *Runtime -> Run all* after opening.


In [ ]:
%pip install -q datasets huggingface_hub pillow matplotlib pandas


In [ ]:
import io
import json
import math
from collections import Counter
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from datasets import load_dataset

# Parquet editions (recommended - single-shot download, no rate limits)
RENDER_REPO   = "gt-free-ocr-metrics/omnidocbench-render-compare-parquet"
LOGPROBS_REPO = "gt-free-ocr-metrics/omnidocbench-qwen-ocr-logprobs-parquet"
# Raw per-page repo retained only for the docsim_triplets config (not yet
# repackaged as parquet - it's already small)
RENDER_REPO_RAW = "gt-free-ocr-metrics/omnidocbench-render-compare"
SAMPLE_N = 4   # pages to display


---
## 1. Render-and-Compare Dataset

`omnidocbench-render-compare-parquet` ships zstd-compressed parquet shards
(~9.4 GB total) containing image bytes and OCR JSON for all five OCR
extraction variants - one row per page.


In [ ]:
VARIANTS = ["ocr_all", "ocr_all_no_mask", "ocr_text", "ocr_formula", "ocr_table"]

rows = []
for v in VARIANTS:
    ds = load_dataset(RENDER_REPO, v, split="train")
    rows.append({"variant": v, "pages": len(ds), "first_page_id": ds[0]["page_id"]})
pd.DataFrame(rows)


### 1.1 Masked original vs reconstructed - side-by-side

The pipeline masks non-target regions on the original scan and renders the OCR
output back to a PNG. Each parquet row holds both PNGs as inline bytes - no
extra file downloads needed.


In [ ]:
ds_all = load_dataset(RENDER_REPO, "ocr_all", split="train")

# Sample pages from different document categories
prefixes = ["PPT", "book", "academic", "newspaper"]
sample_indices = []
for prefix in prefixes:
    for i, pid in enumerate(ds_all["page_id"]):
        if pid.lower().startswith(prefix.lower()):
            sample_indices.append(i)
            break

fig, axes = plt.subplots(len(sample_indices), 2, figsize=(11, 4 * len(sample_indices)))
if len(sample_indices) == 1:
    axes = axes.reshape(1, 2)
for ax_row, idx in zip(axes, sample_indices):
    row = ds_all[idx]
    orig = Image.open(io.BytesIO(row["masked_original"])).convert("RGB")
    recon = Image.open(io.BytesIO(row["reconstructed"])).convert("RGB")
    ax_row[0].imshow(orig);  ax_row[0].set_title(f"masked_original - {row['page_id'][:60]}")
    ax_row[1].imshow(recon); ax_row[1].set_title("reconstructed")
    for ax in ax_row: ax.axis("off")
plt.tight_layout(); plt.show()


### 1.2 Same page across variants

In [ ]:
first_pid = ds_all[0]["page_id"]
compare_variants = ["ocr_all", "ocr_all_no_mask", "ocr_text"]

fig, axes = plt.subplots(1, len(compare_variants), figsize=(15, 5))
for ax, v in zip(axes, compare_variants):
    ds_v = load_dataset(RENDER_REPO, v, split="train")
    matches = [r for r in ds_v if r["page_id"] == first_pid]
    if not matches:
        ax.set_title(f"{v} (no match)"); ax.axis("off"); continue
    row = matches[0]
    img = Image.open(io.BytesIO(row["masked_original"])).convert("RGB")
    ax.imshow(img); ax.set_title(v); ax.axis("off")
plt.tight_layout(); plt.show()


### 1.3 OCR output — HTML and element JSON

In [ ]:
row = ds_all[0]
print("page_id:", row["page_id"])
print()
print("--- ocr_html (first 1500 chars) ---")
print(row["ocr_html"][:1500])


In [ ]:
elements = json.loads(row["ocr_elements"]) if row["ocr_elements"] else []
print(f"Text elements on this page: {len(elements)}")
if elements:
    print("First element:")
    print(json.dumps(elements[0], indent=2, ensure_ascii=False)[:600])


---
## 2. OCR Log-Probabilities Dataset

`omnidocbench-qwen-ocr-logprobs-parquet` provides per-page token-level
log-probabilities and a per-bbox aggregation, both stored inline as JSON
strings. Aggregate statistics are not pre-computed in the parquet schema, so
we derive them on-the-fly from the inline JSON below.


In [ ]:
ds_lp = load_dataset(LOGPROBS_REPO, split="train")
print(f"Rows: {len(ds_lp)}  |  Columns: {ds_lp.column_names}")

def page_features(row):
    try:
        lp = json.loads(row["ocr_logprobs"])
    except Exception:
        return None
    if not lp:
        return None
    logprobs = [t.get("logprob") for t in lp if t.get("logprob") is not None]
    entropies = []
    for t in lp:
        top = t.get("top_logprobs") or []
        ps = [math.exp(x.get("logprob", -50)) for x in top]
        s = sum(ps)
        if s > 0:
            ps = [p/s for p in ps]
            entropies.append(-sum(p*math.log(p) for p in ps if p > 0))
    return {
        "page_id": row["page_id"],
        "n_total_tokens": len(lp),
        "logprob_mean": sum(logprobs)/len(logprobs) if logprobs else 0,
        "logprob_min":  min(logprobs) if logprobs else 0,
        "logprob_max":  max(logprobs) if logprobs else 0,
        "shannon_entropy_mean": sum(entropies)/len(entropies) if entropies else 0,
        "shannon_entropy_max":  max(entropies) if entropies else 0,
    }

# Compute on a 200-row subsample to keep the notebook fast
SUBSET = ds_lp.select(range(min(200, len(ds_lp))))
df_lp = pd.DataFrame([page_features(r) for r in SUBSET if page_features(r)])
print(f"Computed features on {len(df_lp)} pages")
df_lp.head()


### 2.1 Distribution of log-probability and entropy

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df_lp["logprob_mean"], bins=60, color="#4f86c6", edgecolor="white", linewidth=0.5)
axes[0].set_xlabel("mean per-token logprob"); axes[0].set_ylabel("pages")
axes[0].set_title("Per-page mean logprob")

axes[1].hist(df_lp["shannon_entropy_mean"], bins=60, color="#cc6677", edgecolor="white", linewidth=0.5)
axes[1].set_xlabel("mean per-token entropy (nats)"); axes[1].set_ylabel("pages")
axes[1].set_title("Per-page mean entropy")

axes[2].scatter(df_lp["logprob_mean"], df_lp["shannon_entropy_mean"], alpha=0.5, color="#999", s=12)
axes[2].set_xlabel("mean logprob"); axes[2].set_ylabel("mean entropy")
axes[2].set_title("Confidence vs uncertainty")
plt.tight_layout(); plt.show()


### 2.1 Token-count distribution


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df_lp["n_total_tokens"], bins=60, color="#6abf69", edgecolor="white", linewidth=0.5)
ax.set_xlabel("total tokens emitted by OCR per page"); ax.set_ylabel("pages")
ax.set_title("Per-page token count")
plt.tight_layout(); plt.show()


---
## 3. DocSim Training Triplets

The `docsim_triplets` config contains 20,280 triplets used to fine-tune the
DocSim LoRA similarity head. This config currently lives in the **raw**
render-compare repo (the metadata is small enough to bypass rate limits
without repackaging).


In [ ]:
ds_tri = load_dataset(RENDER_REPO_RAW, "docsim_triplets", split="train")
df_tri = ds_tri.to_pandas()
print(f"Total triplets: {len(df_tri)}")
print("Columns:", list(df_tri.columns))
df_tri.head(3)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df_tri["positive_ed"], bins=60, alpha=0.7, label="positive (same page)", color="#4f86c6")
ax.hist(df_tri["negative_ed"], bins=60, alpha=0.7, label="negative (different page)", color="#e07b39")
ax.set_xlabel("Hungarian edit distance  (lower = better OCR fidelity)")
ax.set_ylabel("triplets")
ax.set_title("Edit-distance distribution: positive vs negative pairs")
ax.legend(); plt.tight_layout(); plt.show()


### 3.1 Sample triplets — anchor / positive / negative

In [ ]:
triplet_indices = [0, len(df_tri) // 3, 2 * len(df_tri) // 3]
keys    = ["anchor_path", "positive_path", "negative_path"]
labels  = ["anchor\n(masked original)", "positive\n(same-page recon)", "negative\n(diff-page recon)"]
ed_keys = ["anchor_ed", "positive_ed", "negative_ed"]

fig, axes = plt.subplots(len(triplet_indices), 3, figsize=(15, 5 * len(triplet_indices)))
for row_i, idx in enumerate(triplet_indices):
    row = df_tri.iloc[idx]
    for col_i, (key, label, edk) in enumerate(zip(keys, labels, ed_keys)):
        img = fetch_image(RENDER_REPO, row[key])
        axes[row_i, col_i].imshow(img)
        axes[row_i, col_i].set_title(f"{label}\ned={row[edk]:.3f}", fontsize=9)
        axes[row_i, col_i].axis("off")
plt.suptitle("DocSim triplets", y=1.01)
plt.tight_layout(); plt.show()


---
## 4. Cross-Dataset Exploration

Merging render-and-compare metadata with log-probabilities on `page_id` gives
a unified view: we can examine how the OCR model's internal confidence relates
to document category and output length.


In [ ]:
df_meta = ds_all.to_pandas()[["page_id"]]
df_joined = df_meta.merge(df_lp, on="page_id", how="inner")
print(f"Joined rows: {len(df_joined)}")
df_joined.head()


In [ ]:
def infer_category(pid):
    pid_l = pid.lower()
    for cat, kw in [("Slides/PPT","ppt"),("Book","book"),("Academic paper","academic"),
                    ("Research report","research"),("Financial","financial"),
                    ("Newspaper","news"),("Exam","exam"),("Magazine","magazine")]:
        if kw in pid_l:
            return cat
    return "Other"

df_joined["category"] = df_joined["page_id"].apply(infer_category)

fig, ax = plt.subplots(figsize=(11, 5))
for cat, grp in df_joined.groupby("category"):
    ax.scatter(grp["logprob_mean"], grp["n_total_tokens"],
               label=cat, alpha=0.5, s=12)
ax.set_xlabel("logprob_mean  (Qwen confidence; closer to 0 = higher confidence)")
ax.set_ylabel("n_total_tokens  (OCR output length)")
ax.set_title("Per-page OCR confidence vs output length, coloured by document category")
ax.legend(markerscale=2, fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()
